# 13b — Fundraising → Ballot Mentions: District Explorer

## Question

**How strong is the relationship between fundraising and ballot mentions,
and where does that relationship become less tight?**

This is a small presentation/exploration notebook.

It does three things:

1. shows the **overall fundraising → mentions relationship**;
2. lets us select a district from a dropdown;
3. zooms into **District 4**, where the relationship is weaker and several
   candidates with similar fundraising received very different support.

> This notebook is descriptive. It helps us identify cases to investigate;
> it does not explain *why* a candidate over- or under-performed.


## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

from IPython.display import display, clear_output
import ipywidgets as widgets


# ---------------------------------------------------------------
# Find the repository root
# ---------------------------------------------------------------

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    ROOT = cwd

elif (cwd.parent / "pyproject.toml").exists():
    ROOT = cwd.parent

else:
    raise FileNotFoundError(
        "Could not find pyproject.toml in this folder or its parent."
    )


YEAR = 2024

print("ROOT:", ROOT)


## 2. Load the candidate-level table

We reuse the clean candidate table created in Notebook 10.

That table already contains:

- candidate;
- district;
- fundraising;
- ballot mentions;
- first-place votes;
- viability.


In [ ]:
data_path = (
    ROOT
    / "data"
    / "processed"
    / "finance_analysis"
    / str(YEAR)
    / "question_3"
    / "topline"
    / "candidate_topline_finance_support.csv"
)


analysis = pd.read_csv(
    data_path
)


# We need fundraising + mentions for this notebook.
plot_data = analysis.dropna(
    subset=[
        "fundraising",
        "mentions",
    ]
).copy()


print("Candidates with matched fundraising:", len(plot_data))
print()
print("Candidates by district:")

display(
    plot_data[
        "district"
    ]
    .value_counts()
    .sort_index()
    .rename("candidates")
    .to_frame()
)


## 3. A tiny helper: correlation and R²

For a simple regression with:

`mentions = a + b × fundraising`

the R² is equal to the squared Pearson correlation.

This lets us calculate the two numbers with very simple code.


In [ ]:
def relationship_stats(data):
    # Pearson correlation between fundraising and mentions.
    correlation = data[
        "fundraising"
    ].corr(
        data["mentions"]
    )

    # For one-predictor OLS with an intercept:
    r_squared = correlation ** 2

    return correlation, r_squared


## 4. Overall pattern: fundraising and ballot mentions

This is the first figure for the slide.

Each point is one candidate. Hover over a point to see who it is.
The line is a simple OLS regression using **raw fundraising dollars**.


In [ ]:
overall_r, overall_r2 = relationship_stats(
    plot_data
)


fig_overall = px.scatter(
    plot_data,
    x="fundraising",
    y="mentions",
    hover_name="canonical_candidate",
    hover_data={
        "district": True,
        "fundraising": ":$,.0f",
        "mentions": ":,.0f",
        "first_place_votes": ":,.0f",
        "is_viable": True,
    },
    trendline="ols",
    template="plotly_white",
    labels={
        "fundraising": "Total fundraising ($)",
        "mentions": "Ballot mentions",
    },
    title=(
        "2024 Portland City Council: "
        "fundraising and ballot mentions"
        f"<br><sup>Pearson r = {overall_r:.2f} | "
        f"R² = {overall_r2:.2f}</sup>"
    ),
)


fig_overall.update_traces(
    marker={
        "size": 9,
        "opacity": 0.75,
    }
)


fig_overall.show()


### Finding

The citywide relationship is positive and fairly strong:

> Candidates who raised more money generally received more ballot mentions.

But the next question is more interesting:

**Does that relationship look equally strong in every district?**


## 5. Compare the four districts

In [ ]:
district_rows = []


for district in sorted(
    plot_data["district"].unique()
):
    district_data = plot_data[
        plot_data["district"].eq(
            district
        )
    ]

    r, r2 = relationship_stats(
        district_data
    )

    district_rows.append(
        {
            "district": int(district),
            "candidates": len(
                district_data
            ),
            "pearson_r": r,
            "r_squared": r2,
        }
    )


district_summary = pd.DataFrame(
    district_rows
)


display(
    district_summary.round(
        {
            "pearson_r": 3,
            "r_squared": 3,
        }
    )
)


In [ ]:
fig_districts = px.bar(
    district_summary,
    x="district",
    y="pearson_r",
    text="pearson_r",
    hover_data={
        "candidates": True,
        "r_squared": ":.3f",
    },
    template="plotly_white",
    labels={
        "district": "District",
        "pearson_r": "Pearson correlation",
    },
    title=(
        "Fundraising → mentions relationship "
        "varies across districts"
    ),
)


fig_districts.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside",
)


fig_districts.update_yaxes(
    range=[0, 1]
)


fig_districts.show()


### Finding to look for

In our current results, **District 4 has a weaker fundraising–mentions
relationship than Districts 1 and 3**.

That does **not** mean fundraising is unrelated to support in D4.

It means the pattern is **less tight**, which makes D4 especially useful
for finding candidate-level exceptions.


## 6. Interactive district explorer

Use the dropdown to move between:

- all districts;
- District 1;
- District 2;
- District 3;
- District 4.

The graph is rebuilt from the selected subset, so the trendline and the
correlation always correspond to what you are seeing.


In [ ]:
district_dropdown = widgets.Dropdown(
    options=[
        ("All districts", "All"),
        ("District 1", 1),
        ("District 2", 2),
        ("District 3", 3),
        ("District 4", 4),
    ],
    value="All",
    description="District:",
)


district_output = widgets.Output()


def show_district(selected_district):
    # -----------------------------------------------------------
    # 1. Filter the data
    # -----------------------------------------------------------

    if selected_district == "All":
        data = plot_data.copy()
        title_label = "All districts"

    else:
        data = plot_data[
            plot_data["district"].eq(
                selected_district
            )
        ].copy()

        title_label = (
            f"District {selected_district}"
        )


    # -----------------------------------------------------------
    # 2. Calculate the relationship
    # -----------------------------------------------------------

    r, r2 = relationship_stats(
        data
    )


    # -----------------------------------------------------------
    # 3. Create the Plotly Express scatter
    # -----------------------------------------------------------

    fig = px.scatter(
        data,
        x="fundraising",
        y="mentions",
        hover_name="canonical_candidate",
        hover_data={
            "district": True,
            "fundraising": ":$,.0f",
            "mentions": ":,.0f",
            "first_place_votes": ":,.0f",
            "is_viable": True,
        },
        trendline="ols",
        template="plotly_white",
        labels={
            "fundraising": "Total fundraising ($)",
            "mentions": "Ballot mentions",
        },
        title=(
            f"{title_label}: fundraising and ballot mentions"
            f"<br><sup>Pearson r = {r:.2f} | "
            f"R² = {r2:.2f} | "
            f"n = {len(data)}</sup>"
        ),
    )


    fig.update_traces(
        marker={
            "size": 10,
            "opacity": 0.8,
        }
    )


    fig.show()


def update_district(change):
    with district_output:
        clear_output(
            wait=True
        )

        show_district(
            change["new"]
        )


district_dropdown.observe(
    update_district,
    names="value",
)


display(
    district_dropdown,
    district_output,
)


# Show the initial graph.
with district_output:
    show_district(
        district_dropdown.value
    )


## 7. Find candidate pairs with similar fundraising but different support

For presentation purposes, a very intuitive exception is:

> Two candidates raised almost the same amount of money, but received very
> different numbers of ballot mentions.

We use simple exploratory thresholds:

- fundraising difference ≤ **10%**
- mentions difference ≥ **50%**

These thresholds are not a statistical test. They are a **case-finding tool**.


In [ ]:
MONEY_TOLERANCE = 0.10
MENTION_GAP = 0.50


pair_rows = []


for district in sorted(
    plot_data["district"].unique()
):
    district_data = (
        plot_data[
            plot_data["district"].eq(
                district
            )
        ]
        .reset_index(
            drop=True
        )
    )


    for i in range(
        len(district_data)
    ):
        for j in range(
            i + 1,
            len(district_data),
        ):
            candidate_a = (
                district_data.iloc[i]
            )

            candidate_b = (
                district_data.iloc[j]
            )


            # -----------------------------------------------
            # Difference in fundraising
            # -----------------------------------------------

            larger_money = max(
                candidate_a["fundraising"],
                candidate_b["fundraising"],
            )

            money_gap = (
                abs(
                    candidate_a["fundraising"]
                    - candidate_b["fundraising"]
                )
                / larger_money
            )


            # -----------------------------------------------
            # Difference in mentions
            # -----------------------------------------------

            larger_mentions = max(
                candidate_a["mentions"],
                candidate_b["mentions"],
            )

            mention_gap = (
                abs(
                    candidate_a["mentions"]
                    - candidate_b["mentions"]
                )
                / larger_mentions
            )


            # -----------------------------------------------
            # Keep only interesting pairs
            # -----------------------------------------------

            if (
                money_gap <= MONEY_TOLERANCE
                and mention_gap >= MENTION_GAP
            ):
                pair_rows.append(
                    {
                        "district": int(
                            district
                        ),
                        "candidate_a": candidate_a[
                            "canonical_candidate"
                        ],
                        "candidate_b": candidate_b[
                            "canonical_candidate"
                        ],
                        "fundraising_a": candidate_a[
                            "fundraising"
                        ],
                        "fundraising_b": candidate_b[
                            "fundraising"
                        ],
                        "mentions_a": candidate_a[
                            "mentions"
                        ],
                        "mentions_b": candidate_b[
                            "mentions"
                        ],
                        "fundraising_gap_pct": (
                            money_gap * 100
                        ),
                        "mentions_gap_pct": (
                            mention_gap * 100
                        ),
                    }
                )


interesting_pairs = pd.DataFrame(
    pair_rows
)


interesting_pairs = (
    interesting_pairs
    .sort_values(
        [
            "district",
            "mentions_gap_pct",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


display(
    interesting_pairs.round(
        {
            "fundraising_a": 0,
            "fundraising_b": 0,
            "mentions_a": 0,
            "mentions_b": 0,
            "fundraising_gap_pct": 1,
            "mentions_gap_pct": 1,
        }
    )
)


## 8. District 4: presentation zoom

This is the second figure I would use on the weekly slide.

It shows the same relationship as the citywide graph, but only for D4.

We label four especially intuitive cases:

- Sarah Silkie
- Moses Ross
- Eric Zimmerman
- Tony Morse

The labels are only for storytelling. The trendline still uses **all D4
candidates with matched fundraising**.


In [ ]:
slide_district = 4


d4 = plot_data[
    plot_data["district"].eq(
        slide_district
    )
].copy()


d4_r, d4_r2 = relationship_stats(
    d4
)


fig_d4 = px.scatter(
    d4,
    x="fundraising",
    y="mentions",
    hover_name="canonical_candidate",
    hover_data={
        "fundraising": ":$,.0f",
        "mentions": ":,.0f",
        "first_place_votes": ":,.0f",
        "is_viable": True,
    },
    trendline="ols",
    template="plotly_white",
    labels={
        "fundraising": "Total fundraising ($)",
        "mentions": "Ballot mentions",
    },
    title=(
        "District 4: similar fundraising can produce "
        "very different support"
        f"<br><sup>Pearson r = {d4_r:.2f} | "
        f"R² = {d4_r2:.2f}</sup>"
    ),
)


fig_d4.update_traces(
    marker={
        "size": 10,
        "opacity": 0.75,
    }
)


# ---------------------------------------------------------------
# Label only the cases we want to discuss on the slide.
# ---------------------------------------------------------------

slide_cases = [
    "Sarah Silkie",
    "Moses Ross",
    "Eric Zimmerman",
    "Tony Morse",
]


for candidate_name in slide_cases:
    candidate_row = d4[
        d4[
            "canonical_candidate"
        ].eq(
            candidate_name
        )
    ]


    # Skip gracefully if a name is not present in the current data.
    if candidate_row.empty:
        continue


    candidate_row = (
        candidate_row.iloc[0]
    )


    fig_d4.add_annotation(
        x=candidate_row["fundraising"],
        y=candidate_row["mentions"],
        text=candidate_name,
        showarrow=True,
        arrowhead=2,
        ax=35,
        ay=-30,
    )


fig_d4.show()


## 9. The two D4 stories to have in your notes

### Sarah Silkie vs. Moses Ross

They raised similar amounts, but received dramatically different ballot reach.

**Working qualitative contrast to investigate:**

- Silkie: first-time candidate, City employee, labor-candidate-school /
  organizational network.
- Ross: long party, consulting, and neighborhood political involvement.

A useful question is whether **different kinds of political networks translate
differently into broad voter support**.

---

### Eric Zimmerman vs. Tony Morse

Their fundraising totals are almost identical, but Zimmerman received far more
mentions.

This is useful as a **second quantitative example**, showing that Silkie–Ross
is not the only exception.

We should complete the qualitative comparison before making a stronger claim
about the mechanism.

---

### Slide takeaway

> **Fundraising is an important predictor of ballot support, but it is not a
> sufficient explanation of candidate performance.**

The next research step is to use these exceptions to investigate what finance
does not capture: public visibility, endorsements, organized constituencies,
prior political roles, and other forms of campaign capacity.


## 10. Optional: inspect only District 4 candidate values

This small table is useful when preparing speaker notes.
It is **not** intended to be the final case-log table; that belongs in Notebook 13.


In [ ]:
display(
    d4[
        [
            "canonical_candidate",
            "fundraising",
            "mentions",
            "first_place_votes",
            "is_viable",
        ]
    ]
    .sort_values(
        "mentions",
        ascending=False,
    )
    .round(
        {
            "fundraising": 0,
            "mentions": 0,
            "first_place_votes": 0,
        }
    )
)
